# 多Agent 自定义工作流

来源:
- https://docs.langchain.com/mcp/multi-agent/custom-workflow

本笔记使用 LangGraph StateGraph 编排多Agent自定义工作流，涵盖有向图设计、状态管理、条件边和并行执行。

---
## 1. 概述

LangGraph的`StateGraph`允许开发者以有向图(DAG)方式编排多Agent工作流。每个节点是一个处理步骤，边定义了执行顺序和数据流动。

### 核心概念

| 概念 | 说明 |
|------|------|
| **StateGraph** | 状态化图结构，节点间共享状态 |
| **Node** | 处理单元(Agent、函数、工具) |
| **Edge** | 节点间连接(有序、条件、并行) |
| **State** | 节点间共享的全局状态 |
| **Command** | 节点返回控制指令(goto/update) |

### 边类型

```
1. 普通边: A → B (顺序执行)
2. 条件边: A → {条件} → B 或 C (分支)
3. 并行扇出: A → [B, C, D] (并发)
4. 并行扇入: [B, C, D] → E (同步汇合)
```

---
## 2. 基础 StateGraph

### 定义状态和节点

```python
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, MessagesState, START, END
from langgraph.graph.message import add_messages
from langgraph.types import Command
from typing import Literal

# 自定义状态
class ResearchState(TypedDict):
    topic: str
    research_notes: str
    outline: str
    draft: str
    final_report: str
    feedback: str

# Agent节点函数
def research_agent(state: ResearchState) -> ResearchState:
    """研究阶段：收集信息"""
    result = llm.invoke(f"研究主题 {state['topic']}，输出详细笔记。")
    return {"research_notes": result.content}

def outline_agent(state: ResearchState) -> ResearchState:
    """大纲阶段：生成结构"""
    result = llm.invoke(
        f"根据研究笔记:\n{state['research_notes']}\n生成报告大纲。"
    )
    return {"outline": result.content}

def writing_agent(state: ResearchState) -> ResearchState:
    """写作阶段：撰写初稿"""
    result = llm.invoke(
        f"按照大纲:\n{state['outline']}\n撰写完整报告。"
    )
    return {"draft": result.content}

# 构建图
builder = StateGraph(ResearchState)
builder.add_node("research", research_agent)
builder.add_node("outline", outline_agent)
builder.add_node("writing", writing_agent)
builder.add_edge(START, "research")
builder.add_edge("research", "outline")
builder.add_edge("outline", "writing")
builder.add_edge("writing", END)
graph = builder.compile()

# 执行
result = graph.invoke({"topic": "量子计算的最新进展"})
print(result["draft"])
```

---
## 3. 条件分支与循环

### 条件边 + 循环评审

```python
def review_agent(state: ResearchState) -> Command[Literal["writing", "research", "__end__"]]:
    """评审节点：判断是否需要修改"""
    review = llm.invoke(f"审阅初稿:\n{state['draft']}\n质量标准: 0-10. 是否需要修改?")
    score = extract_score(review.content)
    
    if score < 7:
        feedback = llm.invoke(f"给出改进建议:{state['draft']}")
        return Command(
            goto="writing",
            update={"feedback": feedback.content}
        )
    elif score < 4:
        return Command(
            goto="research",
            update={"feedback": "需要重新研究，方向有偏差。"}
        )
    else:
        return Command(
            goto="finalize",
            update={"final_report": state["draft"]}
        )

builder.add_node("review", review_agent)
builder.add_node("finalize", finalize_agent)
builder.add_edge("writing", "review")
builder.add_conditional_edges(
    "review",
    lambda state: state["final_report"] if "final_report" in state else "writing",
    {
        "writing": "writing",
        "research": "research",
        "finalize": "finalize"
    }
)
```

---
## 4. 并行执行

### 扇出/扇入模式

```python
from langgraph.graph import StateGraph

class ParallelState(TypedDict):
    query: str
    search_results: List[str]
    summary: str

# 并行搜索节点
def search_web(state: ParallelState) -> ParallelState:
    return {"search_results": [web_search(state["query"])]}

def search_docs(state: ParallelState) -> ParallelState:
    return {"search_results": [doc_search(state["query"])]}

def search_code(state: ParallelState) -> ParallelState:
    return {"search_results": [code_search(state["query"])]}

# 汇总节点
def summarizer(state: ParallelState) -> ParallelState:
    combined = "\n\n".join(state["search_results"])
    result = llm.invoke(f"汇总以下搜索结果:\n{combined}")
    return {"summary": result.content}

# 构建图
builder = StateGraph(ParallelState)
builder.add_node("search_web", search_web)
builder.add_node("search_docs", search_docs)
builder.add_node("search_code", search_code)
builder.add_node("summarizer", summarizer)

# 从START并行扇出到三个搜索节点
builder.add_edge(START, "search_web")
builder.add_edge(START, "search_docs")
builder.add_edge(START, "search_code")

# 扇入汇总
builder.add_edge(["search_web", "search_docs", "search_code"], "summarizer")
builder.add_edge("summarizer", END)

graph = builder.compile()
```

> **注意**: 扇入边需所有上游节点完成后方可执行下游节点。

---
## 5. 高级编排模式

### 5.1 嵌套子图

```python
# 子图定义
sub_builder = StateGraph(SubState)
sub_builder.add_node("step1", ...)
sub_builder.add_node("step2", ...)
sub_graph = sub_builder.compile()

# 主图引用子图
main_builder = StateGraph(MainState)
main_builder.add_node("sub_process", sub_graph)
```

### 5.2 动态Agent路由

```python
def supervisor_agent(state: TeamState) -> Command[Literal["researcher", "coder", "analyst", "__end__"]]:
    """监督Agent：动态分配任务"""
    decision = llm.with_structured_output(TaskAssignment).invoke([
        ("system", "你是团队监督，分配任务给最合适的成员。"),
        ("human", state["task"])
    ])
    return Command(goto=decision.next_agent, update={"assigned_to": decision.next_agent})
```

### 5.3 人机协同

```python
from langgraph.checkpoint import MemorySaver

def human_review(state: TeamState) -> Command[Literal["approve", "reject"]]:
    """等待人工审批"""
    response = interrupt({
        "question": "是否批准以下输出？",
        "content": state["draft"]
    })
    if response.get("approved"):
        return Command(goto="approve")
    else:
        return Command(goto="reject", update={"feedback": response["feedback"]})
```

---
## 6. 完整示例：多Agent研究报告生成器

```python
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command
from typing import TypedDict, List, Literal

class ReportState(TypedDict):
    topic: str
    research_notes: List[str]
    outline: str
    sections: List[str]
    draft: str
    quality_score: float
    final: str

def parallel_research(state: ReportState) -> ReportState:
    """并行研究"""
    notes = []
    for source in ["web", "docs", "code"]:
        notes.append(f"[{source}]: {search_source(state['topic'], source)}")
    return {"research_notes": notes}

def outline_generator(state: ReportState) -> ReportState:
    combined = "\n".join(state["research_notes"])
    outline = llm.invoke(f"生成报告大纲:\n{combined}")
    return {"outline": outline.content}

def section_writer(state: ReportState) -> ReportState:
    sections = []
    for section_title in parse_outline(state["outline"]):
        sec = llm.invoke(f"撰写章节 '{section_title}':\n{state['research_notes']}")
        sections.append(sec.content)
    return {"sections": sections, "draft": "\n\n".join(sections)}

def quality_check(state: ReportState) -> Command[Literal["section_writer", "finalize", "parallel_research"]]:
    review = llm.invoke(f"质量评分(0-10):\n{state['draft']}")
    score = extract_score(review.content)
    if score >= 8:
        return Command(goto="finalize", update={"quality_score": score})
    elif score >= 5:
        return Command(goto="section_writer", update={"quality_score": score})
    else:
        return Command(goto="parallel_research", update={"quality_score": score})

# 构建完整图
builder = StateGraph(ReportState)
builder.add_node("parallel_research", parallel_research)
builder.add_node("outline_generator", outline_generator)
builder.add_node("section_writer", section_writer)
builder.add_node("quality_check", quality_check)
builder.add_node("finalize", lambda s: {"final": format_report(s["draft"])})

builder.add_edge(START, "parallel_research")
builder.add_edge("parallel_research", "outline_generator")
builder.add_edge("outline_generator", "section_writer")
builder.add_edge("section_writer", "quality_check")
builder.add_edge("finalize", END)

graph = builder.compile()
result = graph.invoke({"topic": "AI Agent 架构模式"})
```

### 工作流流程

```
[用户输入主题]
      ↓
[并行研究] ──→ [Web] [Docs] [Code]
      ↓
[生成大纲]
      ↓
[逐节写作]
      ↓
[质量检查] ──→ 分数>=8 → [终稿] → 输出
      │
      ├── 分数5-7 → 返回[逐节写作]
      └── 分数<5  → 返回[并行研究]
```

---
## 7. 最佳实践

1. **状态设计**: 明确每个节点需要读取和写入的状态字段
2. **单一职责**: 每个Agent/节点只做一件事
3. **错误处理**: 添加错误捕获和降级节点
4. **可观测性**: 使用LangSmith追踪执行流程
5. **检查点**: 启用checkpointer支持中断恢复
6. **测试**: 对子图单独进行单元测试

## 8. 参考链接

- [LangGraph 自定义工作流](https://docs.langchain.com/mcp/multi-agent/custom-workflow)
- [LangGraph StateGraph 文档](https://langchain-ai.github.io/langgraph/concepts/high_level/)
- [LangGraph 如何指南](https://langchain-ai.github.io/langgraph/how-tos/)
- [LangGraph 检查点文档](https://langchain-ai.github.io/langgraph/concepts/persistence/)
- [Python SDK](https://pypi.org/project/langgraph/)